In [1]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score


In [2]:
import warnings
warnings.filterwarnings("ignore")


In [3]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac1112/doublet_filtered.h5ads/_dataset.h5ads')

In [4]:
df_meta_all = pd.read_csv('/data2st1/junyi/output/atac1112/ATACSC_3REGION_ALL_L2annoated.csv',index_col=0)

In [11]:
df_l3l4 = pd.read_csv('/data2st1/junyi/output/atac1112/iterative/annotated_l3l4.csv',index_col=0)

In [12]:
df_meta_all.head()

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,Sample_name,Condition,Region,celltype.L2.raw,celltype.L2.refined,region_nt,celltype.L3_x,celltype.L4_x,celltype.L3_y,celltype.L4_y
MC37A_AMY:AAACGAAAGAGTGGAA-1,MC37A_AMY,0.092856,0.130307,2,6,0,0,1,1,5,...,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY Meis1_Abi3bp Glut-1,AMY Meis1_Abi3bp Glut-1-0,AMY Meis1_Abi3bp Glut-1,AMY Meis1_Abi3bp Glut-1-0
MC37A_AMY:AAACGAAAGGGTAGTC-1,MC37A_AMY,0.128610,0.014116,18,7,4,5,5,6,7,...,MC37A_AMY,MC,AMY,Microglia-1,Perivascular Macrophage,NN,Microglia-1-1,Microglia-1-1-1,Microglia-1-1,Microglia-1-1-1
MC37A_AMY:AAACGAAGTACGGAGT-1,MC37A_AMY,0.118241,0.019370,4,1,0,0,1,1,1,...,MC37A_AMY,MC,AMY,AMY Maf_Pthlh GABA,AMY Maf_Pthlh GABA,AMY_GABA,NaN,NaN,NaN,NaN
MC37A_AMY:AAACGAAGTCAGCAAG-1,MC37A_AMY,0.074653,0.063387,6,6,0,0,1,1,5,...,MC37A_AMY,MC,AMY,AMY Zfhx4_Pde7b GABA,AMY Zfhx4_Pde7b GABA,AMY_GABA,AMY Zfhx4_Pde7b GABA-1,AMY Zfhx4_Pde7b GABA-1-1,AMY Zfhx4_Pde7b GABA-1,AMY Zfhx4_Pde7b GABA-1-1
MC37A_AMY:AAACGAAGTCCGAGCT-1,MC37A_AMY,0.098286,0.032532,1,8,0,0,1,1,1,...,MC37A_AMY,MC,AMY,AMY Meis1_Abi3bp Glut,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY Meis1_Abi3bp Glut-0,AMY Meis1_Abi3bp Glut-0-0,AMY Meis1_Abi3bp Glut-0,AMY Meis1_Abi3bp Glut-0-0


In [13]:
df_l3l4.head()

,celltype.L3,celltype.L4,celltype.L4.raw
MC37A_AMY:AAAGGATAGCATTGGG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAAACCGTCAGAGTG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAATCGCAGAAAGAG-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-0
MC37A_AMY:ACAGAAAGTAGAATAC-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1
MC37A_AMY:ACAGCGCAGTGAGCAC-1,AMY Rai14_Foxp2 GABA-0,AMY Rai14_Foxp2 GABA-0-0,AMY Rai14_Foxp2 GABA-0-1


In [14]:
df_meta_all = df_meta_all.merge(df_l3l4[['celltype.L3','celltype.L4']],left_index=True,right_index=True,how='left')

In [17]:
%time dmr_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_3regions_ext.bed')


CPU times: user 1h 52min 30s, sys: 19min 18s, total: 2h 11min 49s
Wall time: 6min 39s


In [18]:
dmr_mat.write(f"/data2st1/junyi/output/atac1112/3REGIONS_dmr.h5ads")

... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical


In [19]:
adata_concat.close()

In [26]:
adata_peak = sc.read_h5ad(f"/data2st1/junyi/output/atac1112/3REGIONS_peak.h5ads",backed='r')

# Compute npeaks_on (cells × peaks > 0) and scaled version
Equivalent to `cdr2 <- colSums(assay(sca) > 0); colData(sca)$ngeneson <- scale(cdr2)` for scRNA-seq.

- `npeaks_on`: number of peaks with non-zero insertion count per cell
- `npeaks_on_scaled`: z-score across cells

In [27]:
# Make sure the matrix is fully loaded into memory so we can use sparse operations quickly
# If the file is huge, switch to backed='r' to read chunks, but colSums is convenient on in-memory sparse matrix.
adata_peak_mem = adata_peak#.to_memory() if hasattr(adata_peak, "to_memory") else adata_peak

In [28]:
adata_peak_mem_obs = adata_peak_mem.obs.copy()

In [30]:
import scipy.sparse as sp
from sklearn.preprocessing import scale

# `adata_peak_mem.X` is a snapatac2 _CSRDataset (because of backed='r') and
# does not support direct comparison with an int. Iterate in chunks instead.
X = adata_peak_mem.X
n_obs = adata_peak_mem.shape[0]
npeaks_on = np.zeros(n_obs, dtype=np.int64)

chunk_size = 10_000


In [32]:
for start in range(0, n_obs, chunk_size):
    end = min(start + chunk_size, n_obs)
    chunk = X[start:end]  # snapatac2 returns a scipy.sparse matrix for slicing
    if sp.issparse(chunk):
        npeaks_on[start:end] = np.asarray((chunk > 0).sum(axis=1)).ravel()
    else:
        npeaks_on[start:end] = (chunk > 0).sum(axis=1)

adata_peak_mem_obs["npeaks_on"] = npeaks_on.astype(int)

# Z-score across cells (same as R's base::scale)
npeaks_on_scaled = scale(npeaks_on, with_mean=True, with_std=True)
adata_peak_mem_obs["npeaks_on_scaled"] = npeaks_on_scaled

In [33]:
adata_peak_mem_obs

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,Condition,Region,celltype.L2.raw,region_nt,celltype.L3,celltype.L4,celltype.L2.refined,expriment,npeaks_on,npeaks_on_scaled
MC37A_AMY:AAACGAAAGAGTGGAA-1,MC37A_AMY,0.092856,0.130307,2,6,0,0,1,1,5,...,MC,AMY,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-1,AMY_Meis1_Abi3bp_Glut-1-0,AMY Meis1_Abi3bp Glut,MC,36909,1.884001
MC37A_AMY:AAACGAAAGGGTAGTC-1,MC37A_AMY,0.128610,0.014116,18,7,4,5,5,6,7,...,MC,AMY,Microglia-1,NN,Microglia-1-1,Microglia-1-1-1,Perivascular Macrophage,MC,4715,-1.231900
MC37A_AMY:AAACGAAGTACGGAGT-1,MC37A_AMY,0.118241,0.019370,4,1,0,0,1,1,1,...,MC,AMY,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0,AMY Maf_Pthlh GABA,MC,8024,-0.911638
MC37A_AMY:AAACGAAGTCAGCAAG-1,MC37A_AMY,0.074653,0.063387,6,6,0,0,1,1,5,...,MC,AMY,AMY Zfhx4_Pde7b GABA,AMY_GABA,AMY_Zfhx4_Pde7b_GABA-1,AMY_Zfhx4_Pde7b_GABA-1-1,AMY Zfhx4_Pde7b GABA,MC,28600,1.079813
MC37A_AMY:AAACGAAGTCCGAGCT-1,MC37A_AMY,0.098286,0.032532,1,8,0,0,1,1,1,...,MC,AMY,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-0,AMY_Meis1_Abi3bp_Glut-0-0,AMY Meis1_Abi3bp Glut,MC,15601,-0.178297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MW65A_AMY:TTTGTGTGTTCCTGTC-1,MW65A_AMY,0.145827,0.008648,2,6,0,0,1,1,5,...,MW,AMY,AMY Rai14_Foxp2 GABA,AMY_GABA,AMY_Rai14_Foxp2_GABA-0,AMY_Rai14_Foxp2_GABA-0-0,AMY Rai14_Foxp2 GABA,MW,6619,-1.047621
MW65A_AMY:TTTGTGTTCAACGTGT-1,MW65A_AMY,0.118720,0.026467,6,6,0,0,1,1,5,...,MW,AMY,AMY Foxp2_Penk GABA,AMY_GABA,AMY_Foxp2_Penk_GABA-0,AMY_Foxp2_Penk_GABA-0-1,AMY Foxp2_Penk GABA,MW,16702,-0.071736
MW65A_AMY:TTTGTGTTCATACTTC-1,MW65A_AMY,0.136702,0.013699,4,1,0,0,1,1,1,...,MW,AMY,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0,AMY Maf_Pthlh GABA,MW,10725,-0.650221
MW65A_AMY:TTTGTGTTCTGAGTAC-1,MW65A_AMY,0.128606,0.018868,10,10,5,6,6,7,8,...,MW,AMY,Astrocyte-1,NN,Astrocyte-1-1,Astrocyte-1-1-1,Astrocyte-1,MW,15306,-0.206848


In [34]:
adata_peak_mem_obs[["npeaks_on", "npeaks_on_scaled"]].head()

,npeaks_on,npeaks_on_scaled
MC37A_AMY:AAACGAAAGAGTGGAA-1,36909,1.884001
MC37A_AMY:AAACGAAAGGGTAGTC-1,4715,-1.231900
MC37A_AMY:AAACGAAGTACGGAGT-1,8024,-0.911638
MC37A_AMY:AAACGAAGTCAGCAAG-1,28600,1.079813
MC37A_AMY:AAACGAAGTCCGAGCT-1,15601,-0.178297


In [39]:
adata_peak_mem_obs

,sample,doublet_probability,doublet_score,leiden,leiden_default,leiden_res_0.1,leiden_res_0.2,leiden_res_0.3,leiden_res_0.4,leiden_res_0.5,...,Condition,Region,celltype.L2.raw,region_nt,celltype.L3,celltype.L4,celltype.L2.refined,expriment,npeaks_on,npeaks_on_scaled
MC37A_AMY:AAACGAAAGAGTGGAA-1,MC37A_AMY,0.092856,0.130307,2,6,0,0,1,1,5,...,MC,AMY,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-1,AMY_Meis1_Abi3bp_Glut-1-0,AMY Meis1_Abi3bp Glut,MC,36909,1.884001
MC37A_AMY:AAACGAAAGGGTAGTC-1,MC37A_AMY,0.128610,0.014116,18,7,4,5,5,6,7,...,MC,AMY,Microglia-1,NN,Microglia-1-1,Microglia-1-1-1,Perivascular Macrophage,MC,4715,-1.231900
MC37A_AMY:AAACGAAGTACGGAGT-1,MC37A_AMY,0.118241,0.019370,4,1,0,0,1,1,1,...,MC,AMY,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0,AMY Maf_Pthlh GABA,MC,8024,-0.911638
MC37A_AMY:AAACGAAGTCAGCAAG-1,MC37A_AMY,0.074653,0.063387,6,6,0,0,1,1,5,...,MC,AMY,AMY Zfhx4_Pde7b GABA,AMY_GABA,AMY_Zfhx4_Pde7b_GABA-1,AMY_Zfhx4_Pde7b_GABA-1-1,AMY Zfhx4_Pde7b GABA,MC,28600,1.079813
MC37A_AMY:AAACGAAGTCCGAGCT-1,MC37A_AMY,0.098286,0.032532,1,8,0,0,1,1,1,...,MC,AMY,AMY Meis1_Abi3bp Glut,AMY_Glut,AMY_Meis1_Abi3bp_Glut-0,AMY_Meis1_Abi3bp_Glut-0-0,AMY Meis1_Abi3bp Glut,MC,15601,-0.178297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MW65A_AMY:TTTGTGTGTTCCTGTC-1,MW65A_AMY,0.145827,0.008648,2,6,0,0,1,1,5,...,MW,AMY,AMY Rai14_Foxp2 GABA,AMY_GABA,AMY_Rai14_Foxp2_GABA-0,AMY_Rai14_Foxp2_GABA-0-0,AMY Rai14_Foxp2 GABA,MW,6619,-1.047621
MW65A_AMY:TTTGTGTTCAACGTGT-1,MW65A_AMY,0.118720,0.026467,6,6,0,0,1,1,5,...,MW,AMY,AMY Foxp2_Penk GABA,AMY_GABA,AMY_Foxp2_Penk_GABA-0,AMY_Foxp2_Penk_GABA-0-1,AMY Foxp2_Penk GABA,MW,16702,-0.071736
MW65A_AMY:TTTGTGTTCATACTTC-1,MW65A_AMY,0.136702,0.013699,4,1,0,0,1,1,1,...,MW,AMY,AMY Maf_Pthlh GABA,AMY_GABA,AMY_Maf_Pthlh_GABA-0,AMY_Maf_Pthlh_GABA-0-0,AMY Maf_Pthlh GABA,MW,10725,-0.650221
MW65A_AMY:TTTGTGTTCTGAGTAC-1,MW65A_AMY,0.128606,0.018868,10,10,5,6,6,7,8,...,MW,AMY,Astrocyte-1,NN,Astrocyte-1-1,Astrocyte-1-1-1,Astrocyte-1,MW,15306,-0.206848


In [35]:
# Quick sanity check: distribution
adata_peak_mem_obs["npeaks_on"].describe()

count    176318.000000
mean      17443.191274
std       10332.193747
min         851.000000
25%        9039.000000
50%       16175.000000
75%       24349.000000
max       65503.000000
Name: npeaks_on, dtype: float64

# Merge selected columns into dmr_mat.obs
- `celltype.L2.refined` and `expriment` are pulled from `adata_peak_mem_obs` (which mirrors `adata_peak.obs`).
- `npeaks_on` and `npeaks_on_scaled` were just computed on the same set of cells.
- The join is done by `obs_name` (the index), so the cell barcode order is preserved.

In [40]:
# Sanity check: index alignment between adata_peak_mem_obs and dmr_mat.obs
print("adata_peak_mem_obs index example:", adata_peak_mem_obs.index[:3].tolist())
print("dmr_mat obs index example:    ", dmr_mat.obs.index[:3].tolist())
print("Number of overlapping cells:", adata_peak_mem_obs.index.intersection(dmr_mat.obs.index).size)
print("Total cells in dmr_mat:    ", dmr_mat.n_obs)
print("Total cells in adata_peak: ", adata_peak_mem.n_obs)

adata_peak_mem_obs index example: ['MC37A_AMY:AAACGAAAGAGTGGAA-1', 'MC37A_AMY:AAACGAAAGGGTAGTC-1', 'MC37A_AMY:AAACGAAGTACGGAGT-1']
dmr_mat obs index example:     ['MC37A_AMY:AAACGAAAGAGTGGAA-1', 'MC37A_AMY:AAACGAAAGGGTAGTC-1', 'MC37A_AMY:AAACGAAGTACGGAGT-1']
Number of overlapping cells: 176318
Total cells in dmr_mat:     176318
Total cells in adata_peak:  176318


In [42]:
# Detect which extra metadata columns are present
candidate_cols = ["celltype.L2.refined", "expriment", "experiment", "celltype.L2", "region", "condition","npeaks_on","npeaks_on_scaled"]
present_cols = [c for c in candidate_cols if c in adata_peak_mem_obs.columns]
print("Available columns to merge:", present_cols)

Available columns to merge: ['celltype.L2.refined', 'expriment', 'celltype.L2', 'npeaks_on', 'npeaks_on_scaled']


In [43]:
# Pick the actual two columns requested, but fall back to anything close if missing
meta_cols = []
for col in ["celltype.L2.refined", "celltype.L2.refine", "celltype.L2", "celltype.L2.ref"]:
    if col in adata_peak_mem_obs.columns:
        meta_cols.append(col)
        break
for col in ["expriment", "experiment"]:
    if col in adata_peak_mem_obs.columns:
        meta_cols.append(col)
        break

score_cols = ["npeaks_on", "npeaks_on_scaled"]
merge_cols = meta_cols + score_cols
print("Columns to merge:", merge_cols)

Columns to merge: ['celltype.L2.refined', 'expriment', 'npeaks_on', 'npeaks_on_scaled']


In [44]:
# Build the small DataFrame we will attach to dmr_mat.obs
df_to_merge = (
    adata_peak_mem_obs.loc[:, merge_cols]
    .copy()
)
df_to_merge = df_to_merge.reindex(dmr_mat.obs.index)

In [45]:
df_to_merge

,celltype.L2.refined,expriment,npeaks_on,npeaks_on_scaled
MC37A_AMY:AAACGAAAGAGTGGAA-1,AMY Meis1_Abi3bp Glut,MC,36909,1.884001
MC37A_AMY:AAACGAAAGGGTAGTC-1,Perivascular Macrophage,MC,4715,-1.231900
MC37A_AMY:AAACGAAGTACGGAGT-1,AMY Maf_Pthlh GABA,MC,8024,-0.911638
MC37A_AMY:AAACGAAGTCAGCAAG-1,AMY Zfhx4_Pde7b GABA,MC,28600,1.079813
MC37A_AMY:AAACGAAGTCCGAGCT-1,AMY Meis1_Abi3bp Glut,MC,15601,-0.178297
...,...,...,...,...
MW65A_AMY:TTTGTGTGTTCCTGTC-1,AMY Rai14_Foxp2 GABA,MW,6619,-1.047621
MW65A_AMY:TTTGTGTTCAACGTGT-1,AMY Foxp2_Penk GABA,MW,16702,-0.071736
MW65A_AMY:TTTGTGTTCATACTTC-1,AMY Maf_Pthlh GABA,MW,10725,-0.650221
MW65A_AMY:TTTGTGTTCTGAGTAC-1,Astrocyte-1,MW,15306,-0.206848


In [46]:
for col in merge_cols:
    dmr_mat.obs[col] = df_to_merge[col].values

In [49]:
# Sanity check the merge
dmr_mat.obs[merge_cols]

,celltype.L2.refined,expriment,npeaks_on,npeaks_on_scaled
MC37A_AMY:AAACGAAAGAGTGGAA-1,AMY Meis1_Abi3bp Glut,MC,36909,1.884001
MC37A_AMY:AAACGAAAGGGTAGTC-1,Perivascular Macrophage,MC,4715,-1.231900
MC37A_AMY:AAACGAAGTACGGAGT-1,AMY Maf_Pthlh GABA,MC,8024,-0.911638
MC37A_AMY:AAACGAAGTCAGCAAG-1,AMY Zfhx4_Pde7b GABA,MC,28600,1.079813
MC37A_AMY:AAACGAAGTCCGAGCT-1,AMY Meis1_Abi3bp Glut,MC,15601,-0.178297
...,...,...,...,...
MW65A_AMY:TTTGTGTGTTCCTGTC-1,AMY Rai14_Foxp2 GABA,MW,6619,-1.047621
MW65A_AMY:TTTGTGTTCAACGTGT-1,AMY Foxp2_Penk GABA,MW,16702,-0.071736
MW65A_AMY:TTTGTGTTCATACTTC-1,AMY Maf_Pthlh GABA,MW,10725,-0.650221
MW65A_AMY:TTTGTGTTCTGAGTAC-1,Astrocyte-1,MW,15306,-0.206848


In [48]:
# Coverage diagnostics
for col in merge_cols:
    n_filled = dmr_mat.obs[col].notna().sum()
    print(f"{col}: {n_filled} / {dmr_mat.n_obs} non-null values")

celltype.L2.refined: 176318 / 176318 non-null values
expriment: 176318 / 176318 non-null values
npeaks_on: 176318 / 176318 non-null values
npeaks_on_scaled: 176318 / 176318 non-null values


In [53]:
dmr_mat.obs["ngeneson"] = dmr_mat.obs["npeaks_on_scaled"].values

In [56]:
dmr_mat.X.max()

np.uint32(139)

In [54]:
dmr_mat.write_h5ad(f"/data2st1/junyi/output/atac1112/3REGIONS_dmr.h5ads")